In [1]:
import os
import numpy as np
from PIL import Image, ImageFilter
import openslide
from trident.Converter import AnyToTiffConverter

# 1) Your sweep definitions:
sweep_ranges = {
    "brightness":     (0.0, 30.0),
    "gaussian_noise": (0.0, 20.0),
    "gaussian_blur":  (0.0, 7.0),
    "jpeg":           (100, 90),
}
n_steps = {
    "brightness":     10,
    "gaussian_noise": 10,
    "gaussian_blur":  10,
    "jpeg":           10,
}
step_sizes = {
    name: (max_ - min_) / (n_steps[name] - 1)
    for name, (min_, max_) in sweep_ranges.items()
}

# 2) Custom perturbation functions:
def brightness_perturb(img: Image.Image, delta: float) -> Image.Image:
    import numpy as np
    arr = np.array(img, dtype=np.float32)
    arr += delta if np.random.rand() < 0.5 else -delta
    return Image.fromarray(arr.clip(0, 255).astype(np.uint8))

def gaussian_noise_perturb(img: Image.Image, sigma: float) -> Image.Image:
    import numpy as np
    arr = np.array(img, dtype=np.float32)
    arr += np.random.normal(0, sigma, arr.shape)
    return Image.fromarray(arr.clip(0, 255).astype(np.uint8))

def gaussian_blur_perturb(img: Image.Image, sigma: float) -> Image.Image:
    return img.filter(ImageFilter.GaussianBlur(radius=sigma))

def jpeg_perturb(img: Image.Image, quality: float) -> Image.Image:
    from io import BytesIO
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=int(quality))
    buf.seek(0)
    return Image.open(buf).convert("RGB")

perturb_funcs = {
    "brightness":     brightness_perturb,
    "gaussian_noise": gaussian_noise_perturb,
    "gaussian_blur":  gaussian_blur_perturb,
    "jpeg":           jpeg_perturb,
}

In [2]:
import os
import numpy as np
from PIL import Image, ImageFilter
import openslide
from trident.Converter import AnyToTiffConverter
import pandas as pd
import torch
import h5py
from pathlib import Path
from trident import OpenSlideWSI
from trident.patch_encoder_models import encoder_factory

# ---------------------------------------------------------------------------- #
# 1) SPLITTING FUNCTION
# ---------------------------------------------------------------------------- #
def split_paths_by_size(paths: list[str], n_jobs: int) -> list[list[str]]:
    """
    Greedy “bin‐packing” split of `paths` into `n_jobs` bins, balancing total file size.
    Returns a list of length n_jobs, each a list of paths.
    """
    # 1) get sizes
    ps = [(p, os.path.getsize(p)) for p in paths]
    # 2) sort descending by size
    ps.sort(key=lambda x: x[1], reverse=True)
    # 3) init bins
    bins = [[] for _ in range(n_jobs)]
    bin_sizes = [0]*n_jobs
    # 4) assign each file to the currently smallest bin
    for p, sz in ps:
        i = min(range(n_jobs), key=lambda j: bin_sizes[j])
        bins[i].append(p)
        bin_sizes[i] += sz
    return bins

# ---------------------------------------------------------------------------- #
# 2) PERTURBATION PIPELINE (uses precomputed list of WSIs)
# ---------------------------------------------------------------------------- #
def run_wsi_custom_pipeline(
    tif_paths: list[str],
    target_mag: float,
    out_root: str,
    perturb_funcs: dict[str, callable],
    sweep_ranges: dict[str, tuple[float,float]],
    step_sizes: dict[str, float],
    n_steps: dict[str, int],
    save_png: bool = False
):
    """
    tif_paths     : list of WSI .tif filepaths to process
    target_mag    : extraction magnification (e.g. 2.5)
    out_root      : where to save perturbed tiffs (& pngs if save_png=True)
    perturb_funcs : mapping name→fn(PIL.Image, magnitude)->PIL.Image
    sweep_ranges  : name→(start,end)
    step_sizes    : name→step
    n_steps       : name→number of points
    save_png      : whether to keep the intermediate PNGs
    """
    os.makedirs(out_root, exist_ok=True)

    def extract_to_pil(path):
        slide = openslide.OpenSlide(path)
        base = float(slide.properties.get('openslide.objective-power', 20))
        mpp0 = float(slide.properties.get('openslide.mpp-x', 0.5))
        mpp_target = mpp0 * (base / target_mag)
        lvl = slide.get_best_level_for_downsample(base/target_mag)
        W,H = slide.level_dimensions[lvl]
        img = slide.read_region((0,0), lvl, (W,H)).convert("RGB")
        return img, mpp_target

    for wsi_path in tif_paths:
        base = Path(wsi_path).stem
        print(f"\n[perturb] {base}")
        img, mpp_target = extract_to_pil(wsi_path)

        for name, fn in perturb_funcs.items():
            print(" ->", name)
            start,end = sweep_ranges[name]
            step = step_sizes[name]
            raw = [start + i*step for i in range(n_steps[name])]
            if end < start: raw = raw[::-1]
            if start==0.0 and raw and abs(raw[0])<1e-8: raw=raw[1:]

            pert_dir = os.path.join(out_root, name)
            os.makedirs(pert_dir, exist_ok=True)
            converter = AnyToTiffConverter(job_dir=pert_dir, bigtiff=True)

            for mag in raw:
                pert_img = fn(img, mag)
                png_name = f"{base}_{name}_{mag:.3f}.png"
                png_path = os.path.join(pert_dir, png_name)
                pert_img.save(png_path)

                converter.process_file(png_path, mpp=mpp_target, zoom=1)
                if not save_png:
                    os.remove(png_path)

    print("\nPerturbation pipeline done.")

In [ ]:
# A) Perturbation step in 4 parallel jobs:
# PATH = "../../.."
PATH = "storage"
job_id = 3

wsi_dir  = os.path.join(PATH, "data", "CAMELYON", "test")
all_wsis = sorted(os.path.join(wsi_dir,f) 
                  for f in os.listdir(wsi_dir) if f.lower().endswith(".tif"))
bins     = split_paths_by_size(all_wsis, n_jobs=4)

In [ ]:
# Suppose this is job 2:
run_wsi_custom_pipeline(
    tif_paths      = bins[job_id],
    target_mag     = 2.5,
    out_root       = os.path.join(wsi_dir,"perturbed_images"),
    perturb_funcs  = perturb_funcs,
    sweep_ranges   = sweep_ranges,
    step_sizes     = step_sizes,
    n_steps        = n_steps,
    save_png       = False
)

In [ ]:
import os
from trident.Converter import AnyToTiffConverter

def png_to_tiff_with_trident(
    png_path: str,
    out_dir: str,
    mpp: float,
    zoom: int = 1,
    bigtiff: bool = True
) -> str:
    """
    Convert a single PNG to a tiled TIFF using Trident’s AnyToTiffConverter.

    Parameters:
    -----------
    png_path : str
        Path to the source PNG file.
    out_dir : str
        Directory where the TIFF will be written.
    mpp : float
        Microns‐per‐pixel resolution to embed in the TIFF metadata.
    zoom : int, optional
        Additional scaling factor to apply (1 = no resize).
    bigtiff : bool, optional
        Whether to write a BigTIFF (necessary for very large images).

    Returns:
    --------
    tiff_path : str
        Full path to the generated TIFF file.
    """
    os.makedirs(out_dir, exist_ok=True)
    converter = AnyToTiffConverter(job_dir=out_dir, bigtiff=bigtiff)
    converter.process_file(png_path, mpp=mpp, zoom=zoom)

    base = os.path.splitext(os.path.basename(png_path))[0]
    # Trident writes out "<base>.tif" into out_dir
    tiff_path = os.path.join(out_dir, f"{base}.tif")
    return tiff_path

# Example usage:
png = "perturbed_images/brightness/test_001_brightness_3.333.png"
out = "perturbed_images/brightness"
tif = png_to_tiff_with_trident(png, out_dir=out, mpp=0.25, zoom=1)
print("Wrote TIFF to:", tif)
